# 07 - ARIMA Forecasting vs Seasonal-Naive Baseline

forecast monthly cohort spend, comparing auto-arima against a seasonal-naive baseline. de-synpuf only covers 2008-2010, so there's no real "future" to forecast into -- the only honest way to check which method is actually better is a **backtest**: hold out the last few months, forecast them from what came before, and compare to what really happened.

In [ ]:
import warnings

import duckdb
import numpy as np
import pandas as pd
import pmdarima as pm
import yaml
import matplotlib.pyplot as plt
from pathlib import Path

# pmdarima's stepwise search calls into an older sklearn code path that
# emits a renamed-parameter FutureWarning on every single fit attempt --
# harmless, but drowns out real output during the search
warnings.filterwarnings("ignore", category=FutureWarning)

with open("../configs/data_sources.yaml") as f:
    data_config = yaml.safe_load(f)
with open("../configs/model_config.yaml") as f:
    config = yaml.safe_load(f)

interim_dir = Path("..") / data_config["interim_dir"]
processed_dir = Path("..") / data_config["processed_dir"]

VALUE_COL = "total_paid"  # swap for inpatient_admits / outpatient_visits / carrier_claims
HORIZON = config["forecast"]["horizon_months"]


In [ ]:
# build the monthly cohort-level series (sql/features/monthly_spend.sql) --
# this is the piece notebooks 01-05 didn't need: everything so far was
# cross-sectional (one row per member), this is the first genuinely
# time-indexed table in the pipeline
con = duckdb.connect()
for name in ["inpatient_claims", "outpatient_claims", "carrier_claims"]:
    con.execute(f"""
        create view {name} as
        select * from read_parquet('{interim_dir}/{name}.parquet')
    """)
con.execute(f"""
    create view continuously_enrolled as
    select * from read_parquet('{processed_dir}/cohort.parquet')
""")

monthly = con.execute(Path("../sql/features/monthly_spend.sql").read_text()).df()

out_path = processed_dir / "monthly_spend.parquet"
monthly.to_parquet(out_path, index=False)
print(f"wrote {out_path} ({len(monthly)} months)")
monthly


In [ ]:
# year_month is a yyyymm integer (e.g. 201003) -- turn it into a proper
# monthly Period index so pandas/pmdarima both understand the time axis
monthly["period"] = pd.PeriodIndex(monthly["year_month"].astype(str), freq="M")
monthly = monthly.set_index("period").sort_index()

history_full = monthly[VALUE_COL]
history_full.plot(title=f"monthly {VALUE_COL} -- full history (2008-2010)", marker="o")


**a real, CMS-documented data artifact, not a modeling bug.** every claim type here -- admits, visits, carrier claims, and total spend -- declines steadily through 2010, accelerating sharply toward December (total spend goes from ~$23M/month in 2008-2009 down to ~$3.9M in December 2010, an ~83% drop). This isn't a real drop in care: the cohort is fixed by construction (everyone here is enrolled and alive through all of 2010), so it can't be people leaving the population. CMS's own *DE-SynPUF User Manual* (Table 1) confirms 2010 claim counts are lower across the full population too, attributing it to "attrition from death and some effects of disclosure treatment" -- their privacy-preserving synthesis process, which appears to thin the data more aggressively in the file's final year. Since this cohort already excludes death attrition (alive through all of 2010 by construction), the decline seen here is most likely almost entirely a synthesis artifact, not a claims-processing delay. A real analyst would still need to correct for it before trusting recent-month values; this notebook doesn't, so treat the backtest below as a demonstration of the *method*, not a verdict on how well ARIMA forecasts genuinely complete data.

**backtest split.** the last `HORIZON` months are held out as ground truth; both methods only ever see the months before that when producing their forecast.

In [ ]:
history_train = history_full.iloc[:-HORIZON]
history_actual = history_full.iloc[-HORIZON:]  # held-out ground truth

print(f"training on {len(history_train)} months, backtesting the last {HORIZON}")


In [ ]:
# seasonal-naive baseline: repeat the same month from one year ago
def seasonal_naive_forecast(history, horizon, season_length=12):
    if len(history) >= season_length:
        seasonal_pattern = history.iloc[-season_length:]
    else:
        seasonal_pattern = history
    reps = (horizon // len(seasonal_pattern)) + 1
    return pd.Series(
        (list(seasonal_pattern) * reps)[:horizon],
        index=history_actual.index,
    )

naive_forecast = seasonal_naive_forecast(history_train, HORIZON)
naive_forecast


In [ ]:
# auto-arima picks (p, d, q) x (P, D, Q, m) via stepwise search. with only
# ~30 training months, a seasonal model (m=12) is working with barely 2.5
# cycles -- auto_arima can fail to fit a seasonal model on that little data,
# so this falls back to a non-seasonal search rather than erroring out
arima_config = config["arima"]
try:
    arima_model = pm.auto_arima(
        history_train,
        seasonal=arima_config["seasonal"],
        m=arima_config["m"],
        max_p=arima_config["max_p"],
        max_q=arima_config["max_q"],
        max_d=arima_config["max_d"],
        suppress_warnings=True,
    )
except ValueError:
    print("seasonal fit failed on this little data -- falling back to non-seasonal auto-arima")
    arima_model = pm.auto_arima(
        history_train,
        seasonal=False,
        max_p=arima_config["max_p"],
        max_q=arima_config["max_q"],
        max_d=arima_config["max_d"],
        suppress_warnings=True,
    )

print(arima_model.summary())
arima_forecast = pd.Series(arima_model.predict(n_periods=HORIZON).values, index=history_actual.index)
arima_forecast


In [ ]:
# plot history, the two forecasts, and what actually happened
fig, ax = plt.subplots(figsize=(9, 4))
history_train.tail(24).plot(ax=ax, label="training history")
history_actual.plot(ax=ax, label="actual (held out)", marker="o", color="black")
naive_forecast.plot(ax=ax, label="seasonal naive forecast", linestyle="--", marker="o")
arima_forecast.plot(ax=ax, label="arima forecast", linestyle="--", marker="o")
ax.legend()
ax.set_title(f"backtest -- {VALUE_COL}, last {HORIZON} months")
plt.show()


**which one actually did better?** MAE/RMSE/MAPE against the real held-out values -- this is the actual answer to "is arima worth the extra complexity over the naive baseline," not just a visual impression from the plot.

In [ ]:
def forecast_accuracy(actual, predicted):
    error = actual.values - predicted.values
    return {
        "mae": round(float(np.mean(np.abs(error))), 2),
        "rmse": round(float(np.sqrt(np.mean(error ** 2))), 2),
        "mape": round(float(np.mean(np.abs(error / actual.values)) * 100), 2),
    }

pd.DataFrame(
    [forecast_accuracy(history_actual, naive_forecast), forecast_accuracy(history_actual, arima_forecast)],
    index=["seasonal_naive", "arima"],
)


**the actual forward-looking forecast.** the backtest above answers "which method to trust"; this refits the winner on *all* 36 months (not just the training slice) to produce the forecast that actually extends past the end of the available data, which is what a dashboard would show.

In [ ]:
final_model = pm.auto_arima(
    history_full,
    seasonal=arima_config["seasonal"],
    m=arima_config["m"],
    max_p=arima_config["max_p"],
    max_q=arima_config["max_q"],
    max_d=arima_config["max_d"],
    suppress_warnings=True,
)
future_index = pd.period_range(history_full.index[-1] + 1, periods=HORIZON, freq="M")
future_forecast = pd.Series(final_model.predict(n_periods=HORIZON).values, index=future_index)

fig, ax = plt.subplots(figsize=(9, 4))
history_full.plot(ax=ax, label="history")
future_forecast.plot(ax=ax, label=f"arima forecast (next {HORIZON} months)", linestyle="--", marker="o")
ax.legend()
ax.set_title(f"{VALUE_COL} -- forward forecast beyond available data")
plt.show()


In [ ]:
# save the backtest comparison + the forward forecast for the dashboard
backtest_out = pd.DataFrame({
    "actual": history_actual.values,
    "seasonal_naive": naive_forecast.values,
    "arima": arima_forecast.values,
}, index=history_actual.index.astype(str))
backtest_out.to_parquet(processed_dir / f"backtest_{VALUE_COL}.parquet")

forecast_out = future_forecast.rename("arima_forecast").to_frame()
forecast_out.index = forecast_out.index.astype(str)
forecast_out.to_parquet(processed_dir / f"forecast_{VALUE_COL}.parquet")

print(f"wrote backtest_{VALUE_COL}.parquet and forecast_{VALUE_COL}.parquet")
